# Milestone 4, Section 1.4: repeating the continual-learning simulation with 5 seeds

The single run in `Continual_Learning_Simulation.ipynb` used seed 42. This notebook repeats the same experiment with seeds 42, 1, 2, 3 and 4. The seed decides which training-split users fall into each interval, which rows the replay buffer keeps, and the bootstrap resamples of the floor and the gate. The test split, the drift schedule and the XGBoost settings stay the same. Seed 42 reproduces the original run.

It reports, per step and strategy, the mean and standard deviation of Qini, AUUC and uplift@10% across seeds, how many seeds triggered and promoted an update, and the update cost. It also runs a paired bootstrap on the full test split (200 Poisson resamples) comparing the selected strategy (retrain + replay) with warm start + replay and with retrain on new data only, at steps 3 (+2 sd), 5 (+3 sd) and 6 (clean data again).

Run all cells (about 30-60 minutes on a Colab CPU runtime). Results go to `outputs/continual_seeds/`.

In [ ]:
# SETUP: install the XGBoost version the saved model was written with, and make the pipeline importable.
# The package is looked for (1) next to this notebook, (2) in an uploaded Milestone_4_code.zip,
# (3) in a clone of the GitHub repository (pulled again if it already exists).
%pip install -q xgboost==3.4.1 psutil
import os, sys, glob, zipfile, subprocess
PKG = None
if os.path.isdir("uplift_pipeline"):
    PKG = "."
if PKG is None and glob.glob("Milestone_4_code*.zip"):
    zipfile.ZipFile(sorted(glob.glob("Milestone_4_code*.zip"))[-1]).extractall(".")
    PKG = "Milestone 4"
if PKG is None:
    if os.path.isdir("INSE-6450"):
        subprocess.run(["git", "-C", "INSE-6450", "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/saopa28/INSE-6450.git"], check=True)
    PKG = os.path.join("INSE-6450", "Milestone 4")
if not os.path.isdir(os.path.join(PKG, "uplift_pipeline")):
    raise FileNotFoundError(f"uplift_pipeline not found in {PKG!r}. Push the 'Milestone 4' folder to GitHub, "
                            "or upload Milestone_4_code.zip to the Colab file panel and run this cell again.")
sys.path.insert(0, os.path.abspath(PKG))
import xgboost; print("xgboost", xgboost.__version__, "| pipeline from", os.path.abspath(PKG))

In [ ]:
import numpy as np, pandas as pd
from IPython.display import display
from uplift_pipeline import load_data, split, load_v1, OUT_DIR
from uplift_pipeline.continual import run_seeds, plot_seeds, STRATEGIES

FRAC, DATA = 1.0, None
SEEDS = (42, 1, 2, 3, 4)
OUT = OUT_DIR / "continual_seeds"
df = load_data(frac=FRAC) if DATA is None else load_data(DATA, frac=FRAC)
sp = split(df); del df
v1 = load_v1()
print({k: f"{len(v[2]):,} rows" for k, v in sp.items()})

In [ ]:
res, summ, upd, upd_summ, boots = run_seeds(v1, sp, seeds=SEEDS, out_dir=OUT)

## Mean ± sd across seeds

In [ ]:
pd.set_option("display.width", 250)
def table(metric):
    m = summ.pivot(index=["step", "drift"], columns="strategy", values=f"{metric}_mean")[list(STRATEGIES)]
    s = summ.pivot(index=["step", "drift"], columns="strategy", values=f"{metric}_sd")[list(STRATEGIES)]
    return m.map(lambda v: f"{v:.6f}") + " ± " + s.fillna(0).map(lambda v: f"{v:.6f}")
print("Qini under the current drift"); display(table("qini_after"))
print("Qini on the original test data (retention)"); display(table("qini_original"))
print("Uplift@10% under the current drift"); display(table("uplift10_after"))
print("Seeds (out of", len(SEEDS), ") that triggered / promoted an update")
display(summ.pivot(index=["step", "drift"], columns="strategy", values="seeds_triggered")[list(STRATEGIES)].astype(int).astype(str)
        + " / " + summ.pivot(index=["step", "drift"], columns="strategy", values="seeds_promoted")[list(STRATEGIES)].astype(int).astype(str))

## Update cost across seeds

In [ ]:
display(upd_summ.round(3))

## Paired bootstrap on the test split: selected strategy minus the alternative
A positive gain means retrain + replay ranks better. The interval is the 95% paired bootstrap interval on the full test split; a gain of 0 with an empty interval means both strategies had the same model deployed.

In [ ]:
display(boots.round(6))
print("\\nSeeds where the selected strategy is significantly better (CI above 0) / worse (CI below 0):")
display(boots.groupby(["step", "drift", "other"]).apply(
    lambda g: pd.Series({"better": int((g.ci_lo > 0).sum()), "worse": int((g.ci_hi < 0).sum()),
                         "mean_gain": g.qini_gain.mean()}), include_groups=False))

In [ ]:
fig = plot_seeds(summ, OUT)
print("Files written:", sorted(p.name for p in OUT.iterdir()))